In [1]:
!pip install pillow_heif
!pip install jiwer
!pip install pandas 
!pip install numpy
!pip install transformers
!pip install tokenizers
!pip install pillow 
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install protobuf
!pip install tiktoken
!pip install transformers[sentencepiece] 


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


...

Looking in indexes: https://download.pytorch.org/whl/cu126



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import torch
import pickle
import torch.nn as nn
from tqdm import tqdm
from torch import optim
from torch.utils import data

import tokenizers
import transformers


import jiwer
import numpy as np
import pandas as pd
import pillow_heif
from PIL import Image

from tokenizers import trainers, processors, pre_tokenizers
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from my_transformations import apply_transformation

In [3]:
def get_tokenizer_object(words_iterable):
    """
    Parameters
    ----------
    Iterable of words

    returns
    -------
    a tokenizer object that implements all the necessary
    special tokens additions

    Description
    ------------
    Takes in an iterable of words and returns a HF tokenizer object over them
    that appends all the necessary special tokens

    Note
    -----
    The tokenizer would expect the words that have single spaces between the letters
    example: 'cat' --> 'c a t'
    """
    #get maximum length of words
    max_length = 0
    for word in words_iterable:
        max_length = max(max_length,len(word))
    #adding 2 for sos and eos tokens
    max_length += 2
    #creating words with spaces between letters
    main_iterable_of_words = [' '.join([letter for letter in word]) for word in words_iterable]
    #specifiing tokenizer parameters
    tokenizer_obj = tokenizers.Tokenizer(model=tokenizers.models.WordLevel(unk_token='<unk>'))
    tokenizer_obj.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer_obj.post_processor = processors.TemplateProcessing(single="<sos> $A <eos>", special_tokens=[('<sos>',2), ('<eos>',3)])
    #enabling padding with maximun length
    tokenizer_obj.enable_padding(direction='right', pad_id = 0, pad_token='<pad>', length=max_length)
    #training tokenizer object
    tokenizer_trainer = trainers.WordLevelTrainer(vocab_size=100, special_tokens=['<pad>','<unk>','<sos>','<eos>'])
    tokenizer_obj.train_from_iterator(iterator=main_iterable_of_words, trainer=tokenizer_trainer)
    #transformers tokenizer
    transformers_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object = tokenizer_obj)
    return transformers_tokenizer, max_length

In [4]:
def make_spaces(word):
    """Makes spaces between the letter of a given word"""
    return ' '.join(list(word))

In [5]:
def padding(tensor:torch.tensor, max_length:int, pad_value:int) -> torch.tensor:
    """Given a 1-d tensor and an integer max_length returns 1-d tensor padded to max_length with pad_value"""
    padded_tensor = torch.zeros(max_length).to(dtype=tensor.dtype) + torch.tensor(pad_value).to(dtype=tensor.dtype)
    for i in range(len(tensor)):
        padded_tensor[i] = tensor[i]
    return padded_tensor

In [6]:
def vit_trans(pil_picture, processor, device='cuda'):
    return processor(pil_picture.convert("RGB"), return_tensors="pt").pixel_values[0].unsqueeze(dim=0).to(device=device)

In [7]:
class TextRecDataset(data.Dataset):
    """Implements dataset for Text Recognition with online augmentation"""
    def __init__(self, dataset:pd.DataFrame, tokenizer:transformers.PreTrainedTokenizerFast, max_length:int, pad_id_y:int, pad_id_z:int, word_processor=make_spaces, picture_processor=vit_trans):
        #main parameters
        self.dataset = dataset
        self.max_length = max_length
        self.pad_id_y = pad_id_y
        self.pad_id_z = pad_id_z
        self.tokenizer = tokenizer
        #processors for words and pictures
        self.word_processor = word_processor
        self.picture_processor = picture_processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, indx):
        #getting image path and target name
        data_indx_row = self.dataset.iloc[indx,:]
        image_path = data_indx_row['ImagePath']
        target_name = data_indx_row['Target']
        #maing image to go through augmentations
        main_image = Image.open(image_path).convert("RGB")
        image_transforms = [img.convert("RGB") for img in apply_transformation(main_image)]
        #iterating over obtained images and applying resnet transformations and converting to necessary formats (Tensors)
        images_to_cat = []
        targets_to_cat_y = []
        targets_to_cat_z = []
        for one_image in image_transforms:
            tensor_image = self.picture_processor(one_image).unsqueeze(dim = 0)
            images_to_cat.append(tensor_image)
        #target names to tensors with a given tokenizer
        target_ids = torch.tensor(self.tokenizer(self.word_processor(target_name)).input_ids, dtype=torch.long)
        target_ids_y = padding(target_ids[:-1], max_length=self.max_length, pad_value=self.pad_id_y)
        target_ids_z = padding(target_ids[1:], max_length=self.max_length, pad_value=self.pad_id_z)
        #appending targets
        for _ in range(len(images_to_cat)):
            targets_to_cat_y.append(target_ids_y.unsqueeze(dim=0))
            targets_to_cat_z.append(target_ids_z.unsqueeze(dim=0))
        #returning with concatenation
        return torch.cat(images_to_cat, dim=0), torch.cat(targets_to_cat_y, dim=0), torch.cat(targets_to_cat_z, dim=0)

In [8]:
class ValidationDataset(data.Dataset):
    def __init__(self, dataset:pd.DataFrame):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, indx):
        data_indx_row = self.dataset.iloc[indx,:]
        return Image.open(data_indx_row['ImagePath']), data_indx_row['Target']

In [9]:
#metrics for OCR problem
def total_accuracy(string_array:np.array) -> float:
    """
    parameters
    ----------
    string_array:np.array
        2-dimensional array of shape (N, 2) where N is number of samples
        first column is ground truth strings, second - prediction strings

    returns
    -------
    accuracy:float
        accuracy counted by the number of 100% matchings
    """
    truth_row = string_array[:,0]
    prediction_row = string_array[:,1]
    number_of_matches = (truth_row==prediction_row).astype(np.int64).sum()
    return (number_of_matches/string_array.shape[0]).item()

def cer_metric(string_array:np.array) -> float:
    """
    parameters
    ----------
    string_array:np.array
        2-dimensional array of shape (N, 2) where N is number of samples
        first column is ground truth strings, second - prediction strings

    returns
    -------
    accuracy:float
        average CER metrics over all dataset
    """
    return np.apply_along_axis(lambda x: jiwer.cer(x[0].item(), x[1].item()), axis=1, arr=string_array).mean().item()

In [10]:
#here I also have to add a class of picture distribution for further stratification
def get_dataset(path_to_folder = '/home/luchian/all_data/datasets/Image_data.v4'):
    """
    Parameters
    ----------
    An absolute path to folder with dataset

    returns
    --------
    a dataframe that has the column names [div, image_path, target]
    with corresponding data (div is column that helps devide data)

    Description
    ------------
    Takes in an absolute path to a folder that contains data in the form
    folder_with_target_name[images]
    i.e. each target has a folder of its name with images in it belonging to that target
    """
    #create a dataframe
    main_frame = pd.DataFrame(columns=['Div','ImagePath','Target'])
    #get target folders
    list_of_target_folders = os.listdir(path_to_folder)
    #for each target folder get the images with that target
    for target_name in list_of_target_folders:
        target_data = target_name.strip().split('.')
        images = os.listdir(path_to_folder+'/'+target_name)
        pd_frame = pd.DataFrame({'Div':[target_data[0] for _ in range(len(images))],'ImagePath':[path_to_folder+'/'+target_name+'/'+image_path for image_path in images], 'Target':[target_data[1] for _ in range(len(images))]})
        #ignore index to make shure we have regular indexing order
        main_frame = pd.concat([main_frame, pd_frame], ignore_index=True)
    return main_frame

In [11]:
def get_metrics(val_dataset:data.Dataset, model:nn.Module, tokenizer:transformers.PreTrainedTokenizerFast, inference, processor):
    """
    description
    -----------
    Given a validation dataset and a model returns CER and word accuracy metrics

    parameters
    ----------
    val_dataset:torch.utils.data.dataset
        dataset that returns a tuple of the form (pil image, target)

    model:nn.Module
        Trained model that is going to make predictions

    tokenizer:transformers.PreTrainedTokenizerFast
        tokenzer for the model to decode predictions into the strings

    inference:Callable
        function that takes in model, tokenizer, word transform, picture transform and makes a prediction given the pillow picture
        argument order of inference: pillow picture, model, tokenizer, word transform (defaulted), picture transform (defaulted), device (defaulted), character generation limit

    returns
    -------
    CER metric: float
    Total matches accuracy: float
    """
    #the lists for true and generated string values
    true_strings = []
    generated_strings = []
    #iterating over validation and updating the lists
    for one_sample in tqdm(val_dataset):
        generated_string = inference(one_sample[0].convert("RGB"), model=model, processor=processor, tokenizer=tokenizer, limit=len(one_sample[1]))
        true_string = one_sample[1]
        #appending values
        true_strings.append(true_string)
        generated_strings.append(generated_string)
    #making numpy arrays of true and generated values of shape (N_samples, 2) for calculating metrics
    numpy_true_pred_values = np.array([true_strings, generated_strings]).T
    CER, ACCURACY = cer_metric(numpy_true_pred_values), total_accuracy(numpy_true_pred_values)
    return CER, ACCURACY

In [12]:
class EarlyStopping(object):
    def __init__(self, patience, epoch_multiplier):
        #main parameters
        self.patience = patience
        self.epoch_multiplier = epoch_multiplier
        #in-training parameters
        self.count = 0
        self.criterion_list = []
        #total number of epochs needed for training
        self.total_epochs = None
        #keeping track of max value of anomaly count
        self.max_anomaly = 0

    def add(self, crit_value):
        self.count += 1
        self.criterion_list.append(crit_value)

    def stopping(self):
        anomaly_count = 0
        #counting number of unwanted criterion increases
        for back_ind in range(-1,-(min(self.patience+2,len(self.criterion_list)-1)),-1):
            if self.criterion_list[back_ind] > self.criterion_list[back_ind-1]:
                anomaly_count += 1
            else:
                #updating anomaly count
                self.max_anomaly = max(self.max_anomaly, anomaly_count)
                anomaly_count = 0
                break
        #if the number of anomaly is larger than patience stop else continue
        if anomaly_count > self.patience:
            self.total_epochs = self.count * self.epoch_multiplier
            return True
        else:
            return False

In [13]:
def model_inference(pillow_picture, model, tokenizer, processor, limit):
    """
    Description
    -----------
    Given an image returns generated text
    """
    tens_pic = processor(pillow_picture)
    gen = model.generate(tens_pic, do_sample=True, top_k=50, top_p=0.95, max_length=limit)
    decoded = main_tokenizer.batch_decode(gen, skip_special_tokens=True)[0].split(' ')
    return ''.join(decoded)

In [14]:
pillow_heif.register_heif_opener()

In [15]:
!nvidia-smi

Mon Nov 10 00:45:25 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 531.79                 Driver Version: 531.79       CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                      TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090       WDDM | 00000000:05:00.0  On |                  N/A |
|  0%   33C    P8               17W / 370W|    184MiB / 24576MiB |      2%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [16]:
dataset = get_dataset(path_to_folder=r'C:\Users\user\Downloads\Image_data.v4')
print(dataset.shape)
dataset.head(5)

(10567, 3)


,Div,ImagePath,Target
0,0,C:\Users\user\Downloads\Image_data.v4/0.0mpfor...,0mpformanchesterexchange
1,0,C:\Users\user\Downloads\Image_data.v4/0.100000...,100000000morethanthecorresponding
2,0,C:\Users\user\Downloads\Image_data.v4/0.10note...,10noteontothebarandordersa
3,0,C:\Users\user\Downloads\Image_data.v4/0.10perc...,10percentandwherethehandlingand
4,0,C:\Users\user\Downloads\Image_data.v4/0.15nati...,15nationnatocouncilsomeofhislistenerssaidhewas


In [17]:
#split for train and test using Div column in the set
#when already splitted drop the div column and upddte index
pd_dataset_train = dataset[dataset['Div'] == '0'].drop(['Div'], axis=1).reset_index().drop(['index'], axis=1)
pd_dataset_val = dataset[dataset['Div'] == '1'].drop(['Div'], axis=1).reset_index().drop(['index'], axis=1)
pd_dataset_test = dataset[dataset['Div'] == '2'].drop(['Div'], axis=1).reset_index().drop(['index'], axis=1)

In [18]:
#name of the model
MODEL_NAME = 'microsoft/trocr-small-handwritten'

In [19]:
main_tokenizer, max_length = get_tokenizer_object(pd_dataset_train['Target'])

In [20]:
main_tokenizer

PreTrainedTokenizerFast(name_or_path='', vocab_size=40, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<sos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [21]:
max_length

64

In [22]:
#loading the model
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
main_tokenizer.eos_token_id = main_tokenizer.get_vocab()['<eos>']
main_tokenizer.bos_token_id = main_tokenizer.get_vocab()['<sos>']
processor.tokenizer = main_tokenizer
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME, pad_token_id=0, bos_token_id=2, eos_token_id=3).to(dtype=torch.float16, device='cuda')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-small-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.decoder.get_input_embeddings().padding_idx = 0
model.decoder.resize_token_embeddings(40)

TrOCRScaledWordEmbedding(40, 256, padding_idx=0)

In [24]:
def vit_trans(pil_picture, processor=processor, device='cuda'):
    return processor(pil_picture.convert("RGB"), return_tensors="pt").pixel_values[0].unsqueeze(dim=0).to(device=device)

In [25]:
def train_model(model, train_loader, epoch:int|float, main_optim, scheduler, main_loss, print_every=1, dev='cpu', early_stop=False, patience=10, epoch_multiplier=10, validation_dataset=None, inference=None,
                names=None):
    """
    Given training parameters trains the model
    """
    #prepare early stopping class if needed
    if early_stop:
        early_stopping = EarlyStopping(patience=patience, epoch_multiplier=epoch_multiplier)
    #if epoch is infinity we iterate infinitely
    if type(epoch) == float:
        epoch = 10**10
    #Handling KeyBoardInterrupt exception error
    try:
        train_losses = []
        #Going through epochs
        for ep in range(epoch):
            #updating model distribution probability
            model.train()
            #we are going to save the mean of losses
            epoch_losses = []
            for X,y,z in tqdm(train_loader, desc=f'Going through the loader on epoch_ #{ep+1}'):
                #preparing for forward propogation
                X,y,z = X.to(device = dev).reshape(-1,3,384,384), y.to(device = dev).reshape(-1,y.shape[2]), z.to(device = dev).reshape(-1,z.shape[2])
                main_optim.zero_grad()
                #getting indcies that we are going to get from the model inference
                #reshape prediction and true values to go through loss
                y_pred = model(pixel_values=X, decoder_input_ids=y).logits.transpose(1,2)
                the_loss = main_loss(y_pred, z)
                #backprop and clipping
                the_loss.backward()
                main_optim.step()
                #scheduler step
                if scheduler != None:
                    scheduler.step()
                #keep track of losses
                epoch_losses.append(the_loss.item())
            #for each epoch we append the mean loss over that epoch
            avg_epoch_loss = np.array(epoch_losses).mean().item()
            train_losses.append(round(avg_epoch_loss,5))
            #updating early loss data
            if early_stop:
                if ep%early_stopping.epoch_multiplier == 0:
                    #calculaing and printing metrics
                    cer, accuracy = get_metrics(validation_dataset, model, train_loader.dataset.tokenizer, inference=inference, processor=vit_trans)
                    print(f'Epoch #{ep+1} | Test CER: {cer} | Test Accuracy: {accuracy}', end = '\n\n')
                    early_stopping.add(cer)
                    if early_stopping.stopping():
                        print('Stopped due to early stopping')
                        break
            #deciding when to actually print losses and other data
            if ep%print_every == 0:
                print(f'Epoch #{ep+1} | Train loss: {train_losses[-1]}',end = '\n\n')
                #each time we print data we also save necessary data
                #train losses
                with open(f"C:\\Users\\user\\Downloads\\Data\\{names['train_losses_name']}.pkl", 'wb') as file:
                    pickle.dump(train_losses, file)
                #early stopping object
                if early_stop:
                    with open(f"C:\\Users\\user\\Downloads\\Data\\{names['early_stopping_name']}.pkl", 'wb') as file:
                        pickle.dump(early_stopping, file)
                #optimizer
                torch.save(main_optim.state_dict(), f=f"C:\\Users\\user\\Downloads\\Data\\{names['optimizer_name']}.pth")
                #model weights
                torch.save(model.state_dict(), f = f"C:\\Users\\user\\Downloads\\Data\\{names['model_name']}.pth")
        return train_losses, early_stopping
    except KeyboardInterrupt:
        return train_losses, early_stopping

In [26]:
#training

In [27]:
tokenizer_object, max_length = main_tokenizer, max_length

In [28]:
max_length

64

In [29]:
model.float()

VisionEncoderDecoderModel(
  (encoder): DeiTModel(
    (embeddings): DeiTEmbeddings(
      (patch_embeddings): DeiTPatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DeiTEncoder(
      (layer): ModuleList(
        (0-11): 12 x DeiTLayer(
          (attention): DeiTAttention(
            (attention): DeiTSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
            )
            (output): DeiTSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): DeiTIntermediate(
            (dense): Linear(in_features=384, out_features=1536, bias=True)
        

In [30]:
TorchDatasetTrain = TextRecDataset(pd_dataset_train, tokenizer=tokenizer_object, max_length=max_length, picture_processor=vit_trans, pad_id_y=0, pad_id_z=-100)
TorchDatasetVal = ValidationDataset(pd_dataset_val)
TorchDatasetTest = ValidationDataset(pd_dataset_test)

In [31]:
#train
epoch = float('inf')
batch_size = 9
lr = 1e-5
loader = data.DataLoader(dataset=TorchDatasetTrain, shuffle=True, batch_size=batch_size)
optimizer = optim.AdamW(params=model.parameters(), lr=lr)
#loading optimizer weights
# optimizer.load_state_dict(torch.load('/home/luchian/all_data/ML_models_artifacts/04-11-2025_TextRec_6000_1000_3_LAYER_optimizer_v1.0.pth'))
criterion = nn.CrossEntropyLoss(reduction='mean', label_smoothing=0.01, ignore_index=-100)

In [32]:
names_dict = {
    'train_losses_name':'VIT_losses_v1.0',
    'early_stopping_name':'VIT_early_v1.0',
    'model_name':'VIT_model_v1.0',
    'optimizer_name':'VIT_optimizer_v1.0'
}

In [ ]:
results = train_model(model = model,
                      train_loader = loader,
                      epoch = epoch,
                      main_optim = optimizer,
                      scheduler=None,
                      main_loss = criterion,
                      print_every = 2,
                      dev = 'cuda',
                      names = names_dict,
                      patience=2,
                      epoch_multiplier=2,
                      early_stop=True,
                      validation_dataset=TorchDatasetVal,
                      inference=model_inference)

100%|██████████████████████████████████████████████████████████████████████████████| 1020/1020 [03:58<00:00,  4.28it/s]


Epoch #1 | Test CER: 0.844906212079407 | Test Accuracy: 0.0

Epoch #1 | Train loss: 2.88304



100%|██████████████████████████████████████████████████████████████████████████████| 1020/1020 [03:47<00:00,  4.48it/s]


Epoch #3 | Test CER: 0.3311026922112772 | Test Accuracy: 0.0

Epoch #3 | Train loss: 1.89526



100%|██████████████████████████████████████████████████████████████████████████████| 1020/1020 [03:48<00:00,  4.46it/s]


...

100%|██████████████████████████████████████████████████████████████████████████████| 1020/1020 [03:47<00:00,  4.48it/s]


Epoch #27 | Test CER: 0.12006742121912758 | Test Accuracy: 0.0

Epoch #27 | Train loss: 0.14445



100%|██████████████████████████████████████████████████████████████████████████████| 1020/1020 [03:48<00:00,  4.47it/s]


Epoch #29 | Test CER: 0.11966277036126931 | Test Accuracy: 0.0

Epoch #29 | Train loss: 0.13584



Going through the loader on epoch_ #30:   3%|█                                        | 20/737 [00:40<23:36,  1.98s/it]

In [ ]:
!nvidia-smi